In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    return {
        "tx_id": f"TX{random.randint(1000,9999)}",
        "user_id": f"u{random.randint(1,20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warszawa", "Kraków", "Gdańsk", "Wrocław"]),
        "category": random.choice(["elektronika", "odzież", "żywność", "książki"]),
        "timestamp": datetime.utcnow().isoformat()
    }

while True:
    tx = generate_transaction()
    producer.send("transactions", tx)
    print("SENT:", tx)
    time.sleep(1)


Writing producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję dużych transakcji...")

for msg in consumer:
    tx = msg.value
    if tx["amount"] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | {tx['category']}")
    

Writing consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    group_id='enrich-group',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

def risk(amount):
    if amount > 3000:
        return "HIGH"
    elif amount > 1000:
        return "MEDIUM"
    return "LOW"

for msg in consumer:
    tx = msg.value
    tx["risk_level"] = risk(tx["amount"])
    print(tx)

Writing consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)

msg_count = 0

for msg in consumer:
    tx = msg.value

    store = tx["store"]
    amount = tx["amount"]

    store_counts[store] += 1
    total_amount[store] += amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n=== PODSUMOWANIE ===")
        for s in store_counts:
            avg = total_amount[s] / store_counts[s]
            print(f"{s} | {store_counts[s]} | {total_amount[s]:.2f} | {avg:.2f}")

Writing consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {"count": 0, "sum": 0, "min": float("inf"), "max": 0})

msg_count = 0

for msg in consumer:
    tx = msg.value
    c = tx["category"]
    a = tx["amount"]

    stats[c]["count"] += 1
    stats[c]["sum"] += a
    stats[c]["min"] = min(stats[c]["min"], a)
    stats[c]["max"] = max(stats[c]["max"], a)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n=== CATEGORY STATS ===")
        for k, v in stats.items():
            print(k, v)

Writing consumer_stats.py


In [7]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from collections import defaultdict, deque
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='anomaly_detector'
)

user_transactions = defaultdict(deque)

print("Anomaly detector started...")

for message in consumer:
    event = message.value

    user_id = event['user_id']
    ts = datetime.fromisoformat(event['timestamp'])

    user_transactions[user_id].append(ts)

    while user_transactions[user_id] and (ts - user_transactions[user_id][0]).total_seconds() > 60:
        user_transactions[user_id].popleft()

    if len(user_transactions[user_id]) > 3:
        print(f"ALERT: user {user_id} > 3 transactions in 60s ({len(user_transactions[user_id])})")
        print(event)

Writing consumer_anomaly.py


In [10]:
ls

Untitled.ipynb       consumer_count.py   consumer_filter.py  producer.py
consumer_anomaly.py  consumer_enrich.py  consumer_stats.py
